[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/math_module2/blob/main/ch4_rules.ipynb)

# Chapter 4 — Companion Notebook
## A Toolkit for Slopes
*How to differentiate almost anything without ever touching a difference quotient again*

Three live pictures of the differentiation rules: a **drill** that names which rule a function calls for and shows the worked derivative, a growing rectangle that *proves* the product rule $(fg)'=f'g+fg'$ by watching its two strips, and a chain-rule machine that assembles $f'(g(x))\cdot g'(x)$ one factor at a time.

> **How to use this notebook.** Run every cell from the top (Shift+Enter). Each widget below is live — drag the sliders and watch the picture respond. Requires `ipywidgets` and `matplotlib`.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from ipywidgets import (interact, interactive, fixed, Dropdown, Checkbox,
                        Button, Output, VBox, HBox,
                        FloatSlider, IntSlider, FloatLogSlider)
from IPython.display import HTML, display, Markdown

# Book 2 palette — matches the printed figures in figures/png/
BLUE="#1f4e79"; RED="#c0392b"; GREEN="#2e8b57"; ORANGE="#e08e0b"
PURPLE="#6c3483"; TEAL="#1b9e9e"; GRAY="#7f8c8d"; DARK="#2c3e50"
SHADE="#9fc5e8"; SHADE2="#f4b6ad"; SHADEG="#a9dfbf"; LIGHT="#d6dbdf"

plt.rcParams.update({
    "figure.figsize": (7.2, 4.5), "figure.dpi": 96,
    "font.family": "serif", "mathtext.fontset": "cm", "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": LIGHT, "grid.linewidth": 0.7,
    "lines.linewidth": 2.2, "lines.solid_capstyle": "round",
})

def axes(ax=None, title=None):
    '''A clean axes with a light grid; returns the axes.'''
    if ax is None:
        _, ax = plt.subplots()
    if title:
        ax.set_title(title, color=DARK)
    return ax


## 1. Rule detector

Before you can *use* the rules you have to **see** which one a function is asking for. Pick a function from the dropdown; the blue curve is $f$ and the red curve is its derivative $f'$ (computed by hand, plotted for a reality check). Then tick **reveal** to name the rule(s) at work and read off the worked derivative.

The whole game of this chapter is pattern recognition: a *power* ($x^5$), a *product* (two factors multiplied, $x^2\sin x$), a *quotient* (one thing over another, $\tfrac{x}{x^2+1}$), or a *composition* — a function tucked **inside** another, $(x^2+1)^7$ or $\sin 3x$ — which is the chain rule's territory.

In [2]:
# Each preset carries its f, its hand-computed f', the rule name,
# and the worked steps (LaTeX) revealed on demand.
_PRESETS = {
    'x^5':
        dict(f=lambda x: x**5, df=lambda x: 5*x**4,
             rule='Power rule',
             work=r'$\frac{d}{dx}x^5 = 5x^{5-1} = 5x^4$',
             note='Bring the exponent down, subtract one.'),
    'x^2 * sin(x)':
        dict(f=lambda x: x**2*np.sin(x),
             df=lambda x: 2*x*np.sin(x) + x**2*np.cos(x),
             rule='Product rule',
             work=(r'$\frac{d}{dx}\big[x^2\sin x\big]'
                   r'=(2x)\sin x + x^2(\cos x)$'),
             note="first' \u00b7 second + first \u00b7 second'."),
    '(x^2 + 1)^7':
        dict(f=lambda x: (x**2 + 1)**7,
             df=lambda x: 7*(x**2 + 1)**6 * 2*x,
             rule='Chain rule',
             work=(r'$\frac{d}{dx}(x^2+1)^7'
                   r'=7(x^2+1)^6\cdot 2x = 14x(x^2+1)^6$'),
             note='Outer (power) first, leave the inside alone, '
                  'times the inside derivative.'),
    'x / (x^2 + 1)':
        dict(f=lambda x: x/(x**2 + 1),
             df=lambda x: (1 - x**2)/(x**2 + 1)**2,
             rule='Quotient rule',
             work=(r'$\frac{d}{dx}\frac{x}{x^2+1}'
                   r'=\frac{(1)(x^2+1)-(x)(2x)}{(x^2+1)^2}'
                   r'=\frac{1-x^2}{(x^2+1)^2}$'),
             note='Bottom \u00b7 top\u2032 \u2212 top \u00b7 '
                  'bottom\u2032, all over bottom squared.'),
    'e^x * x':
        dict(f=lambda x: np.exp(x)*x,
             df=lambda x: np.exp(x)*x + np.exp(x),
             rule='Product rule (with $e^x$)',
             work=(r'$\frac{d}{dx}\big[e^x x\big]'
                   r'=e^x\cdot x + e^x\cdot 1 = e^x(x+1)$'),
             note='$e^x$ is its own derivative; the rest is the '
                  'product rule.'),
    'sin(3x)':
        dict(f=lambda x: np.sin(3*x),
             df=lambda x: 3*np.cos(3*x),
             rule='Chain rule',
             work=(r'$\frac{d}{dx}\sin(3x)'
                   r'=\cos(3x)\cdot 3 = 3\cos(3x)$'),
             note='Outer (sine) first, times the inside '
                  'derivative $3$.'),
}

def _detector(which='x^5', reveal=False):
    p = _PRESETS[which]
    # Keep windows where each function is well-behaved on screen.
    hw = 1.6 if which == '(x^2 + 1)^7' else 3.0
    xs = np.linspace(-hw, hw, 600)
    fig, ax = plt.subplots()
    ax.plot(xs, p['f'](xs), color=BLUE, lw=2.4, label=r'$f$')
    ax.plot(xs, p['df'](xs), color=RED, lw=2.0, ls='--',
            label=r"$f'$")
    ax.axhline(0, color=GRAY, lw=0.8)
    ax.set_xlim(-hw, hw)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_title(f'$f(x) = $ {which}', color=DARK)
    ax.legend(loc='upper left', framealpha=0.9)
    plt.show()
    if reveal:
        display(Markdown(f"**Rule:** {p['rule']}"))
        display(Markdown(p['work']))
        display(Markdown(f"*{p['note']}*"))
    else:
        display(Markdown('*Tick **reveal** to name the rule and '
                         'see the worked derivative.*'))

interact(_detector,
         which=Dropdown(options=list(_PRESETS), value='x^5',
                        description='function'),
         reveal=Checkbox(value=False, description='reveal answer'));


interactive(children=(Dropdown(description='function', options=('x^5', 'x^2 * sin(x)', '(x^2 + 1)^7', 'x / (x^…

## 2. Why the product rule has *two* terms

The naive guess $(fg)' = f'g'$ is wrong, and a growing rectangle shows exactly what it misses. Let the width be $f(t)=1+t$ and the height be $g(t)=1+0.6t$, so the **area** is the product $A(t)=f(t)\,g(t)$. Watch the rectangle grow as $t$ advances.

Each frame the area gains three pieces: a **blue** strip from the extra width ($f'\,g$), a **green** strip from the extra height ($f\,g'$), and a tiny **red** corner where both grew at once ($f'g'\,\Delta t$). As the step shrinks that corner vanishes faster than the strips — which is precisely why

$$\frac{d}{dt}(fg) = f'\,g + f\,g'.$$

Two strips survive; the corner does not.

In [3]:
f = lambda t: 1 + t          # width
g = lambda t: 1 + 0.6*t      # height
fp, gp = 1.0, 0.6            # f'(t), g'(t) are constant here

fig, ax = plt.subplots()
_T = np.linspace(0.0, 2.0, 48)   # one t per frame
_dt = 0.55                       # visible growth step per frame

def _frame(i):
    ax.clear()
    t = _T[i]
    w, h = f(t), g(t)
    dw, dh = fp*_dt, gp*_dt
    # Base rectangle: the current area f(t) g(t).
    ax.add_patch(plt.Rectangle((0, 0), w, h, facecolor=LIGHT,
                 edgecolor=DARK, lw=1.5, zorder=1))
    ax.text(w/2, h/2, r'$f\,g$', ha='center', va='center',
            color=DARK, fontsize=14)
    # Strip from extra width: f' g.
    ax.add_patch(plt.Rectangle((w, 0), dw, h, facecolor=SHADE,
                 edgecolor=BLUE, lw=1.2, alpha=0.85, zorder=2))
    ax.text(w + dw/2, h/2, r"$f'g$", ha='center', va='center',
            color=BLUE, fontsize=12, rotation=90)
    # Strip from extra height: f g'.
    ax.add_patch(plt.Rectangle((0, h), w, dh, facecolor=SHADEG,
                 edgecolor=GREEN, lw=1.2, alpha=0.85, zorder=2))
    ax.text(w/2, h + dh/2, r"$fg'$", ha='center', va='center',
            color=GREEN, fontsize=12)
    # The vanishing corner: f' g' (dt)^2.
    ax.add_patch(plt.Rectangle((w, h), dw, dh, facecolor=SHADE2,
                 edgecolor=RED, lw=1.2, zorder=3))
    ax.annotate(r"$f'g'$ (vanishes)", xy=(w + dw/2, h + dh/2),
                xytext=(w + dw/2 + 0.35, h + dh/2 + 0.7),
                color=RED, fontsize=11,
                arrowprops=dict(arrowstyle='->', color=RED))
    ax.set_xlim(0, 5.0); ax.set_ylim(0, 4.0)
    ax.set_aspect('equal')
    ax.set_xlabel(r'width $f(t)=1+t$')
    ax.set_ylabel(r'height $g(t)=1+0.6t$')
    ax.set_title(f'$t = {t:.2f}$:   area $f g = {w*h:.2f}$',
                 color=DARK)

ani = animation.FuncAnimation(fig, _frame, frames=48, interval=90)
plt.close(fig)
HTML(ani.to_jshtml())


## 3. Build a derivative (the chain rule)

Most real functions are **compositions** — an *outer* function wrapped around an *inner* one, $f(g(x))$. Pick an outer and an inner from the dropdowns; the notebook forms the composition $f(g(x))$ and assembles its derivative the way the chain rule tells you to:

$$\frac{d}{dx}\,f(g(x)) = \underbrace{f'(g(x))}_{\text{outer}'\text{ at the inside}}\cdot\;\underbrace{g'(x)}_{\text{inside}'}.$$

Differentiate the outer, **leave the inside alone**, then multiply by the derivative of the inside. The blue curve is the composition $f(g(x))$; the red dashed curve is the derivative the rule produced — so you can confirm the assembled factors are right.

In [4]:
# Outer functions: f, f', and a LaTeX template g(.) -> f(.) and f'(.).
_OUTER = {
    'sin':   dict(f=np.sin, df=np.cos,
                  f_tex=lambda u: rf'\sin\!\left({u}\right)',
                  df_tex=lambda u: rf'\cos\!\left({u}\right)'),
    'exp':   dict(f=np.exp, df=np.exp,
                  f_tex=lambda u: rf'e^{{{u}}}',
                  df_tex=lambda u: rf'e^{{{u}}}'),
    '(.)^3': dict(f=lambda u: u**3, df=lambda u: 3*u**2,
                  f_tex=lambda u: rf'\left({u}\right)^3',
                  df_tex=lambda u: rf'3\left({u}\right)^2'),
    'sqrt':  dict(f=np.sqrt, df=lambda u: 0.5/np.sqrt(u),
                  f_tex=lambda u: rf'\sqrt{{{u}}}',
                  df_tex=lambda u: rf'\frac{{1}}{{2\sqrt{{{u}}}}}'),
}
# Inner functions: g, g', a LaTeX string, and a domain that keeps
# sqrt's argument >= 0 so every composition is real on screen.
_INNER = {
    'x^2':    dict(g=lambda x: x**2, dg=lambda x: 2*x,
                   tex='x^2', dtex='2x', dom=(-2.2, 2.2)),
    '3x + 1': dict(g=lambda x: 3*x + 1, dg=lambda x: 3 + 0*x,
                   tex='3x+1', dtex='3', dom=(0.0, 2.0)),
    'x^2 + 1':dict(g=lambda x: x**2 + 1, dg=lambda x: 2*x,
                   tex='x^2+1', dtex='2x', dom=(-2.2, 2.2)),
}

def _build(outer='sin', inner='x^2'):
    o, n = _OUTER[outer], _INNER[inner]
    a, b = n['dom']
    xs = np.linspace(a, b, 600)
    gx = n['g'](xs)
    comp = o['f'](gx)                       # f(g(x))
    deriv = o['df'](gx) * n['dg'](xs)       # f'(g(x)) * g'(x)

    fig, ax = plt.subplots()
    ax.plot(xs, comp, color=BLUE, lw=2.4, label=r'$f(g(x))$')
    ax.plot(xs, deriv, color=RED, lw=2.0, ls='--',
            label=r"$f'(g(x))\,g'(x)$")
    ax.axhline(0, color=GRAY, lw=0.8)
    ax.set_xlim(a, b)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_title('Composition and its chain-rule derivative',
                 color=DARK)
    ax.legend(loc='upper left', framealpha=0.9)
    plt.show()

    # Assemble the answer factor by factor.
    f_of_g = o['f_tex'](n['tex'])
    outer_prime = o['df_tex'](n['tex'])
    display(Markdown(rf"**Composition:**  $f(g(x)) = {f_of_g}$"))
    display(Markdown(
        r'**Chain rule:** $\;\dfrac{d}{dx}f(g(x))'
        rf'= \underbrace{{{outer_prime}}}_{{\text{{outer}}\,'
        rf"' \text{{ at the inside}}}}"
        rf'\;\cdot\;\underbrace{{{n["dtex"]}}}'
        r"_{\text{inside}\,'}$"))

interact(_build,
         outer=Dropdown(options=list(_OUTER), value='sin',
                        description='outer  $f$'),
         inner=Dropdown(options=list(_INNER), value='x^2',
                        description='inner  $g$'));


interactive(children=(Dropdown(description='outer  $f$', options=('sin', 'exp', '(.)^3', 'sqrt'), value='sin')…

---

*Five rules — power, sum, product, quotient, chain — and you can now differentiate almost anything in seconds, never returning to a difference quotient. The chain rule is the one to keep closest: in Chapter 9 it becomes the engine of gradient descent, and run **backward** across a deep composition it is the backpropagation that trains every neural network.*